In [1]:
import logging
import os
from typing import Optional

import awkward as ak
import h5py
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numba as nb
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score

from utils import get_jets, get_numerical, get_symmetries
from merged import sel_pred_t_by_prob

hep.style.use(hep.style.ROOT)
logging.basicConfig(level=logging.INFO)

PLOT_DIR = os.path.join(os.getcwd(), "multi_roc_curves")
os.makedirs(PLOT_DIR, exist_ok=True)

BASE_TPR = np.linspace(0., 1., 1000)
NTOPS = 2
NJETS = 3*NTOPS + 4
NFJETS = NTOPS + 1

FILES = {
    "tt": {"eval": "/storage/af/user/tsievert/topNet/h5s/tt_hadronic_noBQ.h5", "test": "/storage/af/user/tsievert/topNet/h5s/fjTag_testing.h5"},
    "qcd": {"eval": "/storage/af/user/tsievert/topNet/h5s/qcd_4j_Alltesting_noBQeval.h5", "test": "/storage/af/user/tsievert/topNet/h5s/qcd_4j_wTargets_Alltesting.h5"},
    "t": {"eval": "/storage/af/user/tsievert/topNet/h5s/tt_semileptonic_Alltesting_noBQeval.h5", "test": "/storage/af/user/tsievert/topNet/h5s/ttbar_semileptonic/ttbar_semileptonic.h5"}
    # "4t": "",
}


In [10]:
LOADED_FILES = {label: {filetype: h5py.File(filepath) for filetype, filepath in filepaths.items()} for label, filepaths in FILES.items()}
RECO_CLASSES = list(LOADED_FILES[next(iter(LOADED_FILES))]["eval"]["SpecialKey.Targets"].keys())
SCORES = [item for item in LOADED_FILES[next(iter(LOADED_FILES))]["eval"]["SpecialKey.Targets"][next(iter(RECO_CLASSES))].keys() if "probability" in item]
ASSIGNMENTS = {reco: [item for item in LOADED_FILES[next(iter(LOADED_FILES))]["eval"]["SpecialKey.Targets"][reco].keys() if item not in SCORES] for reco in RECO_CLASSES}
SYMMETRIES = get_symmetries(RECO_CLASSES, ASSIGNMENTS)

ROC_CURVES = {
    # tt performance
    "Correct tt vs. QCD": {'sig': [(True, LOADED_FILES["tt"])], 'bkg': [(False, LOADED_FILES["qcd"])]},
    "Correct tt vs. Incorrect tt": {'sig': [(True, LOADED_FILES["tt"])], 'bkg': [(False, LOADED_FILES["tt"])]},
    # tt vs. t comparison (should be random if theyre equal performance)
    # "tt vs. t": {'sig': [(True, LOADED_FILES["tt"])], 'bkg': [(True, LOADED_FILES["t"])]},
    # t performance
    "Correct t vs. QCD": {'sig': [(True, LOADED_FILES["t"])], 'bkg': [(False, LOADED_FILES["qcd"])]},
    "Correct t vs. Incorrect t": {'sig': [(True, LOADED_FILES["t"])], 'bkg': [(False, LOADED_FILES["t"])]},
    # Everything
    "Correct tt and t vs. QCD and Incorrect tt and t": {'sig': [(True, LOADED_FILES["tt"]), (True, LOADED_FILES["t"])], 'bkg': [(False, LOADED_FILES["qcd"]), (False, LOADED_FILES["tt"]), (False, LOADED_FILES["t"])]},
}

In [3]:
def n_alpha(string: str):
    return len([c for c in string if c.isalpha()])

def find_correct_assignments(reco: str, dataset: dict[str, str]):
    assignments = ASSIGNMENTS[reco]
    correct_assignments = np.ones_like(dataset["eval"]["SpecialKey.Targets"][reco][assignments[0]], dtype=bool)
    for assignment in assignments:
        correct_assignments = np.logical_and(
            correct_assignments,
            dataset["eval"]["SpecialKey.Targets"][reco][assignment][:] == dataset["test"]["TARGETS"][reco][assignment][:]
            if n_alpha(assignment) == 1 else 
            dataset["eval"]["SpecialKey.Targets"][reco][assignment][:] == (dataset["test"]["TARGETS"][reco][assignment][:] + NJETS)
        )
    return correct_assignments

def get_sigORbkg_pred_and_truth(reco: str, score: str, datasets: list[tuple[bool, dict[str, str]]]):
    preds, truths, = [], []
    for correct_assignment, dataset in datasets:
        correct_assignment_mask = (find_correct_assignments(reco, dataset) == correct_assignment)
        all_preds = dataset["eval"]["SpecialKey.Targets"][reco][score][:]
        all_truths = dataset["test"]["TARGETS"][reco]["MASK"][:]
        preds.append(all_preds[correct_assignment_mask]); truths.append(all_truths[correct_assignment_mask])
    return np.concatenate(preds), np.concatenate(truths)

def get_sigANDbkg_preds_and_truths(reco: str, score: str, roc_curve_datasets: dict[str, list[tuple[bool, dict[str, str]]]]):
    signal_preds, signal_truths = get_sigORbkg_pred_and_truth(reco, score, roc_curve_datasets['sig'])
    background_preds, background_truths = get_sigORbkg_pred_and_truth(reco, score, roc_curve_datasets['bkg'])

    return np.concatenate((signal_preds, background_preds)), np.concatenate((signal_truths, background_truths))

In [4]:
def plot_roc(fprs: list[np.ndarray], tprs: list[np.ndarray], labels: list[str], title: Optional[str]=None, xlabel: Optional[str]=None, ylabel: Optional[str]=None, save: Optional[str]=None, show: bool=False):
    if xlabel is None: xlabel = 'Bkg Eff.'
    if ylabel is None: ylabel = 'Sig Eff.'

    plt.figure(figsize=(10, 8))
    for fpr, tpr, label in zip(fprs, tprs, labels):
        plt.plot(fpr, tpr, label=label)
    plt.legend()
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xscale('log')
    plt.yscale('log')
    if title is not None: plt.title(title)
    if save is not None: plt.savefig(os.path.join(PLOT_DIR, save))
    if show: plt.show()
    else: plt.close()


In [ ]:
for roc_curve_label, roc_curve_datasets in ROC_CURVES.items():
    for reco in RECO_CLASSES:
        try:
            fprs, tprs, labels = [], [], []
            for score in SCORES:
                preds, truths = get_sigANDbkg_preds_and_truths(reco, score, roc_curve_datasets)
                fpr, tpr, thresholds = roc_curve(truths, preds)
                auc = roc_auc_score(truths, preds)
                fprinterp = np.interp(BASE_TPR, tpr, fpr)
                thresholdsinterp = np.interp(BASE_TPR, tpr, thresholds)
                fprs.append(fprinterp); tprs.append(BASE_TPR), labels.append(score.replace('_probability', '')+f' - AUC = {auc:.3f}')
            plot_roc(fprs, tprs, labels, title=roc_curve_label+' - '+reco, save='_'.join(roc_curve_label.split(' '))+'_'+reco+'.png')
        except:
            print(f'Error creating ROC curve for \'{roc_curve_label}\' with class \'{reco}\'')
            continue
            

/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. QCD' with class 'FBt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. QCD' with class 'FRt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


...

Error creating ROC curve for 'Correct t vs. Incorrect t' with class 'FBt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. Incorrect t' with class 'FRt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. Incorrect t' with class 'SRqqt2'


In [5]:
# A look up table is in shape
# [event x valid_predjets,
#        [retrieved, pt]]
def generate_score_LUT(predicted_jets, target_jets, predicted_scores, target_scores, selected_order):
    return generate_one_pred_LUT(
        predicted_jets, target_jets, predicted_scores, target_scores,
        selected_order, SYMMETRIES,
        ak.ArrayBuilder()
    ).snapshot()

@nb.njit
def generate_one_pred_LUT(
    predicted_jets, target_jets, predicted_scores, target_scores,
    selected_order, symmetries,
    builder
):
    # for each event
    for pjets_event, tjets_event, pscore_event, tscore_event, order_event in zip(
        predicted_jets, target_jets,
        predicted_scores, target_scores,
        selected_order
    ):
        # for each prediction per event, in order of best probs
        for pred_idx in order_event:
            pjets, pscore = pjets_event[pred_idx], pscore_event[pred_idx]
            if pjets is None: continue

            retrieved = 0
            # check all targets of matching reco (i.e. account for symmetry of top label exchange)
            target_idxs = [pred_idx - i for i in range(1, (pred_idx % N_TOPS)+1)][::-1]+[pred_idx]+[pred_idx + i for i in range(1, N_TOPS-(pred_idx % N_TOPS))]
            for targ_idx in target_idxs:
                tjets, tscore = tjets_event[targ_idx], tscore_event[targ_idx]
                if tjets is None: continue

                # check all valid labels (i.e. account for symmetry of jet labels)
                for symand in symmetries[pred_idx]:
                    n_matched = 0
                    for labels in symand:
                        if match_jet(pjets[labels[0]], tjets[labels[1]]): n_matched += 1
                    if n_matched == len(symand): retrieved = 1; break

                if retrieved: break

            builder.begin_list()
            builder.append(retrieved)
            builder.append(pscore)
            builder.append(tscore)
            builder.end_list()

    return builder

In [13]:
def get_jet4moms(dataset: dict[str, str]):
    # jet 4-momentums
    jets = ak.from_regular(ak.zip({
        "pt": np.array(dataset['test']["INPUTS"]["Jets"]["pt"]),
        "eta": np.array(dataset['test']["INPUTS"]["Jets"]["eta"]),
        "phi": np.array(dataset['test']["INPUTS"]["Jets"]["phi"]),
        "mass": np.array(dataset['test']["INPUTS"]["Jets"]["mass"])
    },  with_name="Momentum4D"))
    jets["index"] = ak.local_index(jets)
    N_AK5_JETS = ak.max(ak.local_index(jets), axis=None) + 1
    print(f"Number of AK5 jets: {N_AK5_JETS}")
    fatjets = ak.from_regular(ak.zip({
        "pt": np.array(dataset['test']["INPUTS"]["BoostedJets"]["fj_pt"]),
        "eta": np.array(dataset['test']["INPUTS"]["BoostedJets"]["fj_eta"]),
        "phi": np.array(dataset['test']["INPUTS"]["BoostedJets"]["fj_phi"]),
        "mass": np.array(dataset['test']["INPUTS"]["BoostedJets"]["fj_mass"])
    }, with_name="Momentum4D"))
    fatjets["index"] = ak.local_index(fatjets) + N_AK5_JETS
    N_AK8_JETS = ak.max(ak.local_index(fatjets), axis=None) + 1
    print(f"Number of AK8 jets: {N_AK8_JETS}")
    return jets, fatjets

def get_merged_selections(dataset: dict[str, str]):
    jets, fatjets = get_jet4moms(dataset)
    predicted_jets = get_jets(dataset['eval'], RECO_CLASSES, ASSIGNMENTS, jets, fatjets, targets_key="SpecialKey.Targets")
    predicted_pts = ak.Array(ak.sum(predicted_jets, axis=-1).pt)
    predicted_dps = get_numerical(dataset['eval'], "detection_probability", RECO_CLASSES, targets_key="SpecialKey.Targets")
    predicted_aps = get_numerical(dataset['eval'], "assignment_probability", RECO_CLASSES, targets_key="SpecialKey.Targets")
    _, _, merged_selections = sel_pred_t_by_prob(predicted_jets, predicted_pts, predicted_dps, predicted_aps)

    return merged_selections

def get_merged_predictions(score: str, merged_selections: ak.Array, dataset: dict[str, str]):
    jets, fatjets = get_jet4moms(dataset)
    predicted_jets = get_jets(dataset['eval'], RECO_CLASSES, ASSIGNMENTS, jets, fatjets, targets_key="SpecialKey.Targets")[merged_selections]
    predicted_scores = get_numerical(dataset['eval'], score, RECO_CLASSES, targets_key="SpecialKey.Targets")[merged_selections]

    return predicted_jets, predicted_scores

def get_merged_targets(score: str, merged_selections: ak.Array, dataset: dict[str, str]):
    jets, fatjets = get_jet4moms(dataset)
    target_jets = get_jets(dataset['test'], RECO_CLASSES, ASSIGNMENTS, jets, fatjets)[merged_selections]
    target_score = get_numerical(dataset['test'], "MASK", RECO_CLASSES)[merged_selections]

    return target_jets, target_scores

def get_merged_sigORbkg_pred_and_truth(score: str, datasets: list[tuple[bool, dict[str, str]]]):
    preds, truths, = [], []
    for correct_assignment, dataset in datasets:
        merged_selections = get_merged_selections(dataset)
        predicted_jets, predicted_scores = get_merged_predictions(score, merged_selections, dataset)
        target_jets, target_scores = get_merged_targets(score, merged_selections, dataset)
        scoreLUT = generate_score_LUT(predicted_jets, target_jets, predicted_scores, target_scores, merged_selections)

        print(ak.type(scoreLUT))
        scoreLUT = ak.flatten(scoreLUT, axis=1)
        print(ak.type(scoreLUT))
        correct_assignment_mask = (scoreLUT[:, 0] == correct_assignment)
        preds.append(scoreLUT[:, 1][correct_assignment_mask]); truths.append(scoreLUT[:, 2][correct_assignment_mask])
    return np.concatenate(preds), np.concatenate(truths)

def get_merged_sigANDbkg_preds_and_truths(score: str, roc_curve_datasets: dict[str, list[tuple[bool, dict[str, str]]]]):
    signal_preds, signal_truths = get_merged_sigORbkg_pred_and_truth(score, roc_curve_datasets['sig'])
    background_preds, background_truths = get_merged_sigORbkg_pred_and_truth(score, roc_curve_datasets['bkg'])

    return np.concatenate((signal_preds, background_preds)), np.concatenate((signal_truths, background_truths))

In [14]:
for roc_curve_label, roc_curve_datasets in ROC_CURVES.items():
    fprs, tprs, labels = [], [], []
    for score in SCORES:
        preds, truths = get_merged_sigANDbkg_preds_and_truths(score, roc_curve_datasets)
        
        fpr, tpr, thresholds = roc_curve(truths, preds)
        auc = roc_auc_score(truths, preds)
        fprinterp = np.interp(BASE_TPR, tpr, fpr)
        thresholdsinterp = np.interp(BASE_TPR, tpr, thresholds)
        fprs.append(fprinterp); tprs.append(BASE_TPR), labels.append(score.replace('_probability', '')+f' - AUC = {auc:.3f}')
    plot_roc(fprs, tprs, labels, title=roc_curve_label+' - Merged', save='_'.join(roc_curve_label.split(' '))+'_Merged'+'.png')

Number of AK5 jets: 10
Number of AK8 jets: 3


TypeError: Encountered a None value, but None conversion/promotion is disabled

This error occurred while calling

    ak.to_layout(
        None
        allow_record = False
        regulararray = False
        primitive_policy = 'error'
    )

In [48]:
for files in LOADED_FILES.values(): 
    for file in files.values(): file.close()